# 03 - The Code Mentor Agent

In this notebook, we build a **Code Mentor** agent - an AI assistant that helps developers understand and improve their code through explanation, review, and best practices guidance.

## What You'll Learn

- How to build an agent that analyzes and explains code
- Using specialized subagents for security, performance, and architecture analysis
- Creating custom slash commands for common developer workflows
- Implementing output styles for different experience levels

## Prerequisites

- Familiarity with the basics covered in notebooks 00-02
- Understanding of the Claude Code SDK patterns

In [ ]:
from dotenv import load_dotenv
from utils.agent_visualizer import print_activity, visualize_conversation

from claude_code_sdk import ClaudeCodeOptions, ClaudeSDKClient, query

load_dotenv()

## Basic Usage: Ask About Code

Let's start with a simple query asking the agent to explain a concept. The Code Mentor agent is configured with tools for reading code (`Read`, `Glob`, `Grep`), running analysis (`Bash`), and looking up best practices (`WebSearch`).

In [ ]:
messages = []
async for msg in query(
    prompt="Explain the SOLID principles in software design with Python examples",
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        allowed_tools=["WebSearch"],
    ),
):
    print_activity(msg)
    messages.append(msg)

print(f"\nResult:\n{messages[-1].result if hasattr(messages[-1], 'result') else 'No result'}")

## Code Review: Analyzing Real Code

Now let's use the agent to analyze actual code. We'll point it at a file and ask for a review. The agent can read files, search for patterns, and provide structured feedback.

In [ ]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        cwd="code_mentor_agent",
        system_prompt="""You are a Code Mentor - an expert software engineering guide.
        Help developers understand and improve their code through clear explanations
        and constructive feedback.""",
        allowed_tools=["Read", "Glob", "Grep"],
    )
) as mentor:
    await mentor.query("Review the agent.py file. Focus on code quality and suggest improvements.")
    async for msg in mentor.receive_response():
        print_activity(msg)
        messages.append(msg)

In [ ]:
visualize_conversation(messages)

## Using Specialized Subagents

The Code Mentor has three specialized subagents:

1. **security-reviewer** - Identifies vulnerabilities using OWASP guidelines
2. **performance-analyst** - Analyzes algorithm complexity and optimization opportunities  
3. **architecture-advisor** - Provides guidance on design patterns and code structure

These subagents are defined in `.claude/agents/` and can be invoked via the `Task` tool.

In [ ]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        cwd="code_mentor_agent",
        system_prompt="""You are a Code Mentor with access to specialized subagents:
        - security-reviewer: For security vulnerability analysis
        - performance-analyst: For optimization opportunities
        - architecture-advisor: For design patterns and structure
        
        Use the Task tool to delegate to these subagents when deep analysis is needed.""",
        allowed_tools=["Read", "Task", "Glob", "Grep"],
    )
) as mentor:
    await mentor.query(
        """Analyze agent.py using the architecture-advisor subagent. 
        Focus on the design patterns used and suggest improvements."""
    )
    async for msg in mentor.receive_response():
        print_activity(msg)
        messages.append(msg)

In [ ]:
visualize_conversation(messages)

## Using the Standalone Agent

The Code Mentor agent is packaged as a standalone Python module in `code_mentor_agent/agent.py`. It provides several convenience functions:

- `send_query()` - General-purpose queries
- `explain_code()` - Explain a specific file
- `review_code()` - Review code for issues
- `suggest_improvements()` - Get improvement suggestions

In [ ]:
from code_mentor_agent.agent import send_query, explain_code, review_code, suggest_improvements

In [ ]:
# Explain code with different detail levels
result = await explain_code(
    file_path="agent.py",
    cwd="code_mentor_agent",
    detail_level="brief"
)
print(f"Brief explanation:\n{result}")

In [ ]:
# Review code with a specific focus
result = await review_code(
    file_path="agent.py",
    cwd="code_mentor_agent",
    focus="style"
)
print(f"Style review:\n{result}")

## Output Styles: Adapting to Experience Level

The Code Mentor supports different output styles to adapt explanations to the developer's experience level:

- **beginner** - Patient explanations with foundational concepts and analogies
- **expert** - Concise, technical communication with minimal explanation

These are defined in `.claude/output-styles/` and can be activated via the `settings` option.

In [ ]:
# Beginner-friendly explanation
result = await send_query(
    prompt="What is dependency injection and why should I use it?",
    output_style="beginner"
)
print(f"Beginner explanation:\n{result}")

In [ ]:
# Expert-level explanation
result = await send_query(
    prompt="What is dependency injection and why should I use it?",
    output_style="expert"
)
print(f"Expert explanation:\n{result}")

## Custom Slash Commands

The Code Mentor includes custom slash commands for common workflows:

| Command | Description |
|---------|-------------|
| `/explain <file>` | Get a detailed explanation of code |
| `/review <file>` | Perform a code review with findings |
| `/improve <file> [goal]` | Get actionable improvement suggestions |
| `/security-scan <file>` | Deep security analysis |

These commands are defined in `.claude/commands/` and provide structured prompts for consistent output.

## Agent Structure

The Code Mentor agent follows the recommended structure for Claude Code SDK agents:

```
code_mentor_agent/
├── .claude/
│   ├── agents/                    # Specialized subagents
│   │   ├── security-reviewer.md
│   │   ├── performance-analyst.md
│   │   └── architecture-advisor.md
│   ├── commands/                  # Custom slash commands
│   │   ├── explain.md
│   │   ├── review.md
│   │   ├── improve.md
│   │   └── security-scan.md
│   ├── output-styles/             # Experience-level adaptations
│   │   ├── beginner.md
│   │   └── expert.md
│   └── settings.local.json        # Local settings
├── agent.py                        # Main agent implementation
└── CLAUDE.md                       # Persistent context
```

This structure enables:
- **Modularity**: Each subagent handles a specific domain
- **Reusability**: Slash commands provide consistent workflows
- **Adaptability**: Output styles adjust to user needs

## Conclusion

The Code Mentor agent demonstrates how to build a practical developer tool using the Claude Code SDK. Key takeaways:

1. **Specialized subagents** allow deep analysis in specific domains (security, performance, architecture)
2. **Custom slash commands** provide consistent, structured workflows
3. **Output styles** adapt communication to different experience levels
4. **Convenience functions** make the agent easy to integrate into development workflows

### Next Steps

- Try the Code Mentor on your own codebase
- Create additional subagents for your specific needs (e.g., testing, documentation)
- Add custom slash commands for your team's workflows
- Explore adding hooks for automated code quality tracking